In [2]:
import sys
# Installing the core Transformer libraries without restrictive version locks
!{sys.executable} -m pip install transformers==4.42.4 torch==2.4.1 regex

  Using cached transformers-4.42.4-py3-none-any.whl.metadata (43 kB)
  Using cached torch-2.4.1-cp311-cp311-manylinux1_x86_64.whl.metadata (26 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 289.0 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 159.8 kB/s eta 0:00:0000:0100:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.1/797.1 MB 16.4 kB/s eta 0:00:0000:0107:04��━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━ 439.5/797.1 MB 69.9 kB/s eta 1:25:15��━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━ 621.3/797.1 MB 14.6 kB/s eta 3:20:39��━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━ 621.4/797.1 MB 14.5 kB/s eta 3:22:34��━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━ 621.4/797.1 MB 14.5 kB/s eta 3:22:34
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 245.2 kB/s eta 0:00:0000:0101:09
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 214.1 kB/s eta 0:00:0000:0100:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 322.6 kB/s eta 0:00:0000:0100:02
   ━━━━━━━━━━━━━━━━━━

In [3]:
import re
from transformers import pipeline

# 1. Setup the NLP Pipeline
# Using a general NER model as a baseline
ner_pipe = pipeline("ner", model="dslim/bert-base-NER", aggregation_strategy="simple")

sample_report = """
ALERT: Suspicious activity from IP 192.168.1.100. 
Possible C2 traffic with malicious domain evil-c2.com. 
Linked to 'ShadowBrokers' group. Monitor 10.0.0.5 and malicious-site.org.
"""

# 2. Transformer Logic: Find General Entities
# It identifies the 'ShadowBrokers' as an Organization (ORG)
print("--- Transformer Insights ---")
entities = ner_pipe(sample_report)
for ent in entities:
    print(f"Entity: {ent['word']} | Type: {ent['entity_group']}")

# 3. Regex Logic: Precise Technical Indicators
# Regex is better for rigid patterns like IPs
ip_pattern = r'\b(?:\d{1,3}\.){3}\d{1,3}\b'
domain_pattern = r'\b[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}\b'

ips = set(re.findall(ip_pattern, sample_report))
# Filter to ensure we only get valid domains
domains = set([d for d in re.findall(domain_pattern, sample_report) if '.' in d])

print(f"\n--- Extracted IOCs ---\nIPs: {ips}\nDomains: {domains}")

2026-03-09 19:22:54.768049: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-09 19:22:55.231276: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-03-09 19:22:55.231800: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-09 19:22:55.299646: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-03-09 19:22:55.591186: I tensorflow/core/platform/cpu_feature_guar

config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

--- Transformer Insights ---
Entity: IP | Type: MISC
Entity: ShadowBrokers | Type: ORG

--- Extracted IOCs ---
IPs: {'192.168.1.100', '10.0.0.5'}
Domains: {'evil-c2.com', 'malicious-site.org'}
